# 06 — SHAP Analysis

Explain model predictions using SHAP (SHapley Additive exPlanations).
- Global feature importance
- SHAP summary plots
- Single-match explanations
- Feature interaction effects

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from mls_predictor.data_loader import load_raw_data
from mls_predictor.elo import compute_elo_history
from mls_predictor.feature_engine import build_all_features, get_feature_columns
from mls_predictor.model_utils import (
    temporal_train_test_split, prepare_features, train_all_models,
)
from mls_predictor.shap_utils import global_feature_importance

In [2]:
%%time
df = load_raw_data()
df = compute_elo_history(df)
df_feat = build_all_features(df)
feature_cols = get_feature_columns()['all']

split = temporal_train_test_split(df_feat, 'target_1x2', feature_cols)
X_train, X_test, imputer, scaler = prepare_features(split['X_train'], split['X_test'])
models = train_all_models(X_train, split['y_train'].values, '1x2')

CPU times: total: 1min 49s
Wall time: 1min 29s


In [3]:
# Global feature importance (mean |SHAP|)
for model_name in ['xgboost', 'lightgbm', 'random_forest']:
    if model_name in models:
        imp_df = global_feature_importance(
            models[model_name], X_test[:200], split['feature_cols']
        )
        if imp_df is not None:
            print(f'\n═══ {model_name.upper()} — Top 15 Features ═══')
            print(imp_df.head(15).to_string(index=False))

ValueError: Per-column arrays must each be 1-dimensional

In [ ]:
# SHAP Summary Plot (beeswarm) for XGBoost
if 'xgboost' in models:
    explainer = shap.TreeExplainer(models['xgboost'])
    shap_values = explainer.shap_values(X_test[:300])
    
    # For multi-class, pick class 2 (Home Win)
    if isinstance(shap_values, list) and len(shap_values) > 2:
        print('SHAP summary for Home Win (class=2):')
        shap.summary_plot(
            shap_values[2], X_test[:300],
            feature_names=split['feature_cols'],
            max_display=20, show=True
        )
    else:
        shap.summary_plot(
            shap_values, X_test[:300],
            feature_names=split['feature_cols'],
            max_display=20, show=True
        )

In [ ]:
# Single-match SHAP waterfall
if 'xgboost' in models:
    explainer = shap.TreeExplainer(models['xgboost'])
    # Pick a random test match
    idx = 0
    test_row = split['test_df'].iloc[idx]
    print(f"Match: {test_row['HomeTeam']} vs {test_row['AwayTeam']} "
          f"({int(test_row['FTHG'])}-{int(test_row['FTAG'])})")
    
    explanation = explainer(X_test[idx:idx+1])
    # For multi-class, pick predicted class
    pred = models['xgboost'].predict(X_test[idx:idx+1])[0]
    class_names = ['Away', 'Draw', 'Home']
    print(f"Predicted: {class_names[int(pred)]}")
    
    shap.plots.waterfall(explanation[0, :, int(pred)], max_display=15, show=True)

In [ ]:
# Feature importance comparison across models
importance_data = {}
for name in ['xgboost', 'lightgbm', 'random_forest']:
    if name in models:
        imp = global_feature_importance(models[name], X_test[:200], split['feature_cols'])
        if imp is not None:
            importance_data[name] = imp.set_index('feature')['mean_abs_shap']

if importance_data:
    combined = pd.DataFrame(importance_data)
    combined['avg'] = combined.mean(axis=1)
    top20 = combined.sort_values('avg', ascending=False).head(20)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top20.drop('avg', axis=1).plot(kind='barh', ax=ax)
    ax.set_title('SHAP Feature Importance — Model Comparison', fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()